In [2]:
import sys
sys.path.insert(0, '../src')

from features.rolling import (
    build_team_match_history,
    add_rolling_features,
    add_venue_splits,
    add_extended_rolling_features,
    add_days_rest,
    add_new_team_imputation,
    build_match_level_features,
)

In [3]:
import pandas as pd

# Handcrafted dataset:
# TeamA has strong prior home form and poor prior away form before match 5
matches = pd.DataFrame(
    {
        "match_id": [1, 2, 3, 4, 5],
        "date": [
            "2024-01-01",
            "2024-01-08",
            "2024-01-15",
            "2024-01-22",
            "2024-01-29",
        ],
        "home_team": ["TeamA", "TeamB", "TeamA", "TeamC", "TeamA"],
        "away_team": ["TeamB", "TeamA", "TeamC", "TeamA", "TeamD"],
        "home_goals": [2, 1, 3, 2, 1],
        "away_goals": [0, 0, 1, 0, 1],
    }
)

history = build_team_match_history(matches)
history = add_rolling_features(history, window_size=2)
history = add_venue_splits(history, window_size=2)
history = add_extended_rolling_features(history, window_size=2)
history = add_days_rest(history)
history = add_new_team_imputation(history)

match_features = build_match_level_features(history)

pd.set_option("display.max_columns", None)

# Look at the target match
target = match_features[match_features["match_id"] == 5][
    [
        "match_id",
        "home_team",
        "away_team",
        "home_rolling_win_rate_home",
        "home_rolling_win_rate_away",
        "away_rolling_win_rate_home",
        "away_rolling_win_rate_away",
    ]
]

print(target.to_string(index=False))

# Explicit acceptance checks
row = target.iloc[0]

print("\nAcceptance Test 3 checks:")
print("home_rolling_win_rate_home exists:", "home_rolling_win_rate_home" in match_features.columns)
print("home_rolling_win_rate_away exists:", "home_rolling_win_rate_away" in match_features.columns)
print("away_rolling_win_rate_home exists:", "away_rolling_win_rate_home" in match_features.columns)
print("away_rolling_win_rate_away exists:", "away_rolling_win_rate_away" in match_features.columns)

print("\nShould differ for TeamA in match 5:")
print("home split value:", row["home_rolling_win_rate_home"])
print("away split value:", row["home_rolling_win_rate_away"])
print("Values differ:", row["home_rolling_win_rate_home"] != row["home_rolling_win_rate_away"])

 match_id home_team away_team  home_rolling_win_rate_home  home_rolling_win_rate_away  away_rolling_win_rate_home  away_rolling_win_rate_away
        5     TeamA     TeamD                         1.0                         0.0                         1.0                         0.0

Acceptance Test 3 checks:
home_rolling_win_rate_home exists: True
home_rolling_win_rate_away exists: True
away_rolling_win_rate_home exists: True
away_rolling_win_rate_away exists: True

Should differ for TeamA in match 5:
home split value: 1.0
away split value: 0.0
Values differ: True
